**Human Movement Intelligence | Multimodal Movement Learning | Presented by: Reza Saadatyar (2026)**

**01 | NinaPro DB2 Source Pairing and Label Audit**

![Original S01 E1 recordings, source-level sensor pairing, corrected label audit and evidence exports](../../images/01_ninapro_pairing_and_label_audit.svg)

**Research question:** Does the NinaPro DB2 S01 exercise-one (E1) recording provide row-aligned sEMG and acceleration measurements, corrected movement labels and identifiable repetition boundaries for paired-window analysis?

**Dataset:** [NinaPro DB2](https://ninapro.hevs.ch/instructions/DB2.html), participant S01, exercise E1 (`DB2_s1/S1_E1_A1.mat`). The downloaded subject archive is **448.47 MiB** and contains three exercise-specific MATLAB files.

**Observed source:** **1,808,331 released sample rows**, **12 sEMG channels**, **36 acceleration columns**, **18 corrected labels (0–17)** and **205 contiguous corrected-label/repetition segments**.

**Notebook scope:** Inspect source integrity and MATLAB variables, audit row alignment and corrected annotations, assess finite-value coverage and visualise an original two-second interval. The analysis describes the released recordings; it neither fits a classifier nor independently measures native sensor synchronisation.


**00. Colab Repository Setup**

In [1]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


/content/human-movement-intelligence


**01. Imports and Research Environment**

In [2]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("multimodal_movement_learning")

# ----------------- 2. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 3. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

plot_style(
    font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(9, 4),
)

# ------------------------ 4. Study Research Utilities ------------------------
from ninapro_source import (
    RELEASED_RATE_HZ, ensure_subject_archive,
    list_mat_members, load_exercise, inspect_record, contiguous_segments,
)

# ---------------------- 5. Source and Output Locations -----------------------
study_dir = root / "multimodal_movement_learning"
output_dir = study_dir / "output"
processed_dir = root / "data" / "processed" / "ninapro_db2"
raw_dir = root / "data" / "raw" / "ninapro_db2"
output_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
print("Research output directory:", output_dir)


Repository up to date.
Research output directory: /content/human-movement-intelligence/multimodal_movement_learning/output


**02. Acquire One Original Participant Archive**

The source reader reuses the S01 archive when it is already available locally. Otherwise, it downloads the official preprocessed NinaPro DB2 ZIP and checks its integrity before reading the exercise entries. The inspected archive contains three MATLAB recordings, one each for E1, E2 and E3.

This notebook uses E1 because it is the selected finger-movement exercise. A new, temporary Colab runtime may need to retrieve the original archive again if its local data cache is no longer available.


In [3]:
# =============== 02. Acquire One Original Participant Archive ================

NOTEBOOK_NUMBER = 1
PILOT_SUBJECT = 1
EXERCISE_ID = 1
DOWNLOAD_MISSING = True

source = ensure_subject_archive(
    raw_dir, PILOT_SUBJECT, download=DOWNLOAD_MISSING,
)
print("Subject source:", source.name)
print("Archive size (MiB):", round(source.stat().st_size / (1024**2), 2))
print("MATLAB entries:", list_mat_members(source))


Subject source: DB2_s1.zip
Archive size (MiB): 448.47
MATLAB entries: ['DB2_s1/S1_E1_A1.mat', 'DB2_s1/S1_E2_A1.mat', 'DB2_s1/S1_E3_A1.mat']


**03. Read Paired MATLAB Measurements**

Read the E1 `emg` and `acc` measurement arrays together with their original and corrected annotation fields. The observed shapes are **(1,808,331, 12)** for sEMG and **(1,808,331, 36)** for acceleration.

These modalities are associated by the **same released sample-row indices**, rather than being paired retrospectively by movement label. The source audit also records exercise identity, channel dimensions, corrected class codes and repetition identifiers. This establishes the stored-data schema used for later windowing.


In [4]:
# ==================== 03. Read Paired MATLAB Measurements ====================

record = load_exercise(source, PILOT_SUBJECT, exercise=EXERCISE_ID)
audit = pd.DataFrame([inspect_record(record)])
display(audit.T.rename(columns={0: "observed value"}))
print("EMG shape:", record.emg.shape, "| Acc shape:", record.acc.shape)
print("Released source-sample rate (Hz):", RELEASED_RATE_HZ)


,observed value
subject_id,S01
exercise_id,1
source_file,DB2_s1/S1_E1_A1.mat
samples,1808331
duration_s_nominal,904.1655
emg_channels,12
acc_columns,36
emg_finite_fraction,1.0
acc_finite_fraction,1.0
corrected_labels,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17"


EMG shape: (1808331, 12) | Acc shape: (1808331, 36)
Released source-sample rate (Hz): 2000.0


**04. Inspect the Released Time Base and Row Alignment**

Compare the number of rows in sEMG, acceleration, `restimulus` and `rerepetition`. All four S01 E1 arrays contain **1,808,331 rows**, corresponding to approximately **904.17 seconds** on the released **2,000-Hz common grid**.

The accelerometers were originally sampled at approximately **148 Hz** and represented on the common released grid. Matching stored rows therefore supports index-based pairing, but does not demonstrate independent 2,000-Hz accelerometer measurements or quantify native sensor-clock offsets.


In [5]:
# ============= 04. Inspect Time Base and Source-Level Alignment ==============

rows = len(record.emg)
alignment_summary = pd.DataFrame([{
    "subject_id": record.subject_id,
    "emg_rows": rows,
    "acc_rows": len(record.acc),
    "corrected_label_rows": len(record.restimulus),
    "repetition_rows": len(record.rerepetition),
    "nominal_duration_s": rows / RELEASED_RATE_HZ,
    "native_acc_sampling_hz_published": 148,
}])
assert alignment_summary[["emg_rows", "acc_rows", "corrected_label_rows", "repetition_rows"]].nunique(axis=1).eq(1).all()
display(alignment_summary)


,subject_id,emg_rows,acc_rows,corrected_label_rows,repetition_rows,nominal_duration_s,native_acc_sampling_hz_published
0,S01,1808331,1808331,1808331,1808331,904.1655,148


**05. Audit Corrected Labels and Repetitions**

The corrected movement target is `restimulus`, with source codes **0 (Rest)** and **1–17 (movement classes)**. S01 E1 contains **1,064,608 Rest rows (58.87%)**, so class support is not balanced across the recorded timeline.

Combining corrected labels with `rerepetition` identifies **205 contiguous segments**. Repetition codes **0–6** occur; each non-rest movement class has **six segments** in the inspected participant. The repetition field is segmentation metadata, not a predictor.

The published corrected-boundary procedure uses information from sEMG. This annotation provenance matters when later comparing sEMG-only and acceleration-only models. The reported counts describe S01, not an assumed fixed pattern for every participant.


In [6]:
# ================ 05. Audit Corrected Labels and Repetitions =================

labels, counts = np.unique(record.restimulus, return_counts=True)
label_dictionary = pd.DataFrame({
    "exercise_id": EXERCISE_ID,
    "label": labels.astype(int),
    "source_samples": counts.astype(int),
    "duration_s_nominal": counts / RELEASED_RATE_HZ,
})
segments = contiguous_segments(record.restimulus, record.rerepetition)
segment_frame = pd.DataFrame(segments, columns=[
    "start_sample", "end_sample", "label", "repetition_id",
])
segment_frame["duration_s"] = (segment_frame.end_sample - segment_frame.start_sample) / RELEASED_RATE_HZ
display(label_dictionary)
print("Corrected repetitions:", sorted(np.unique(record.rerepetition).tolist()))
print("Source label segments:", len(segment_frame))
display(segment_frame.groupby("label", as_index=False).agg(segments=("label", "size"),
    median_duration_s=("duration_s", "median")))


,exercise_id,label,source_samples,duration_s_nominal
0,1,0,1064608,532.3040
1,1,1,78701,39.3505
2,1,2,58979,29.4895
3,1,3,50985,25.4925
4,1,4,45065,22.5325
5,1,5,40125,20.0625
6,1,6,25557,12.7785
7,1,7,58155,29.0775
8,1,8,28238,14.1190
9,1,9,39903,19.9515


Corrected repetitions: [0, 1, 2, 3, 4, 5, 6]
Source label segments: 205


,label,segments,median_duration_s
0,0,103,5.22350
1,1,6,6.45850
2,2,6,4.78925
3,3,6,4.23500
4,4,6,3.90850
5,5,6,3.43825
6,6,6,2.17275
7,7,6,4.83150
8,8,6,2.39850
9,9,6,3.41725


**06. Inspect Signal Quality and a Paired Source Interval**

Evaluate finite-value coverage for both modalities and plot the **first two seconds** of sEMG channel 1 and acceleration column 1 using their shared released sample indices. Both S01 E1 source arrays contain **100% finite entries**.

The traces retain their original source values, with time expressed in seconds. They illustrate two stored channels over the same interval; this visual comparison does not estimate inter-device latency or summarise waveforms across participants.


In [ ]:
# =========== 06. Inspect Real Signal Quality and Temporal Context ============

quality = pd.DataFrame({
    "modality": ["sEMG", "Acceleration"],
    "channels": [record.emg.shape[1], record.acc.shape[1]],
    "finite_fraction": [np.isfinite(record.emg).mean(), np.isfinite(record.acc).mean()],
})
display(quality)
length = min(len(record.emg), int(RELEASED_RATE_HZ * 2))
t_s = np.arange(length) / RELEASED_RATE_HZ
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True, layout="constrained")
axes[0].plot(t_s, record.emg[:length, 0], lw=0.8)
axes[0].set(ylabel="sEMG channel 1 (source units)", title="S01 E1: paired source interval (first 2 s)")
axes[1].plot(t_s, record.acc[:length, 0], lw=0.9)
axes[1].set(xlabel="Time (s)", ylabel="Acc column 1 (source units)")
for ax in axes:
    ax.grid(alpha=0.2)
save_figure(fig, output_dir / "01_alignment_examples.png")
plt.show()


**07. Construct Source, Label and Sensor Dictionaries**

Create three complementary records for subsequent analysis:

- **Pairing registry:** participant and exercise identity, original source reference, row count, channel dimensions and observed data coverage.
- **Label dictionary:** corrected E1 class identifiers, their sample counts and nominal durations.
- **Sensor dictionary:** the 12 sEMG channels and 36 acceleration columns, grouped by the twelve tri-axial sensor modules.

These tables describe the source and its annotations without copying the original signal arrays. Participant and repetition identifiers are retained for segmentation and evaluation, not used as input features for the movement classifiers.


In [8]:
# ============= 07. Export Source, Label and Sensor Dictionaries ==============

sensors = ([{"modality": "sEMG", "column": f"emg_{i+1:02d}",
             "sensor_module": i+1, "axis": "sEMG"} for i in range(12)]
           + [{"modality": "Acceleration", "column": f"acc_{i*3+j+1:02d}",
               "sensor_module": i+1, "axis": "xyz"[j]} for i in range(12) for j in range(3)])
sensor_dictionary = pd.DataFrame(sensors)
audit.to_csv(output_dir / "01_pairing_registry.csv", index=False)
label_dictionary.to_csv(output_dir / "01_label_dictionary.csv", index=False)
sensor_dictionary.to_csv(output_dir / "01_sensor_dictionary.csv", index=False)
display(sensor_dictionary.groupby("modality", as_index=False).agg(columns=("column", "count")))


,modality,columns
0,Acceleration,36
1,sEMG,12


**08. Research Summary and Output Download**

The source audit confirms matching released row counts, finite paired measurements and an identifiable corrected-label/repetition inventory for S01 E1. These results support subsequent source-based windowing; they do not establish native sensor synchronisation precision.

Notebook 02 uses the pairing registry and corrected-label dictionary to inspect the wider E1 cohort before constructing aligned windows and participant-disjoint evaluation assignments.

The final cell packages the **four current research outputs** and displays their filenames once, together with the Colab download location.


In [ ]:
# ================= 08. Research Summary and Output Download ==================

download_outputs_from_colab(
    folder="multimodal_movement_learning/output",
    stage=f"{NOTEBOOK_NUMBER:02d}",
)
